JAMfit Getting Started: Reconstructing One Filament in a Torus with Ports {#doc_tCurr_jamfit_one_filament}
==========
This is the smallest and simplest JAMfit example. It walks you through making synthetic magnetics data, then reconstructs the original setup from that data and checks the result against the truth (the inputs used to generate the synthetic data). It runs in a few minutes.

**What this shows**
1. Building a sensor set from scratch (a poloidal array of Mirnov probes) and setting up JAMfit.
2. Making synthetic data with the full ThinCurr wall model (`gen_synthetic_data`).
3. Building a small reduced wall model from the dominant eigenmodes (`create_from_runTD_top_modes`).
4. Reconstructing the filament current and the wall currents at each time with `run_reconstruction_lstsq`, and choosing the Ip constraint weight.

**The setup**
- **Wall:** the ThinCurr torus with ports from \ref doc_tCurr_hodlr (major radius 1 m, minor radius 0.5 m, 10 rectangular ports).
- **Coils** (from `oft_in.xml`), three wires in all: an upper coil at (R, Z) = (1.0, 0.7) m and a lower coil at (1.0, -0.7) m, each with a constant 1 kA, and one plasma filament at (1.0, 0) m that ramps up to 1 MA and then quenches.
- **TokaMaker mesh:** `one_filament_tokamaker_mesh.h5`, a 2D cross-section of the same torus (the wall circle as the limiter, plus the two coils). JAMfit always sets up TokaMaker; here it is only used to plot the geometry. The DIII-D VDE example (\ref doc_tCurr_jamfit_vde_create) shows the TokaMaker post-processing (LCFS, q95, li).

## Load ThinCurr library

To load the ThinCurr python module we need to tell python where to the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environement variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` for binaries on macOS).

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6, 4)
%matplotlib inline

# Use the OFT install pointed to by OFT_ROOTPATH, if set
oft_python_path = os.getenv('OFT_ROOTPATH')
if oft_python_path is not None:
    sys.path.append(os.path.join(oft_python_path, 'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr.JAMfit import JAMfit
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov

## User Inputs and Setting up JAMfit
In the following cells we define the initial inputs for creating a JAMfit class object and create the JAMfit object.

JAMfit class objects require the following inputs:

| Input | Type | Description |
|---|---|---|
| `xml_file` | str | ThinCurr XML configuration file (non-filament coils first, then filaments) |
| `thincurr_meshfile` | str | ThinCurr wall mesh file |
| `tokamaker_meshfile` | str | 2D TokaMaker mesh file of the same device |
| `B0` | float | Vacuum toroidal field at `R0` [T] |
| `R0` | float | Reference major radius [m] |
| `nthreads` | int | Threads for a new `OFT_env` (needed if `oft_env` is not given) |
| `oft_env` | OFT_env | Existing OFT environment to reuse (optional) |

**Note:** All file names start with `one_filament_` so they do not clash with the DIII-D example in the same folder.

In [ ]:
xml_file = 'oft_in.xml'                                  # ThinCurr coils: [upper coil, lower coil, plasma filament]
thincurr_meshfile = '../ports/thincurr_ex-ports.h5'      # ThinCurr wall mesh, shared with the ports example
tokamaker_meshfile = 'one_filament_tokamaker_mesh.h5'    # 2D TokaMaker mesh of the same torus
floops_file = 'one_filament_floops.loc'                  # sensor file written below
reduced_filename = 'one_filament_reduced_model.h5'       # reduced wall model written below

B0 = 1.0          # vacuum toroidal field at R0 [T]; only sets F0 = R0*B0 in TokaMaker, which this example does not use
R0 = 1.0          # major radius of the torus [m]

num_non_fil_coils = 2   # coils listed before the plasma filament in oft_in.xml
num_modes = 2           # wall eigenmodes kept in the reduced model (see 'Build the reduced wall model')
initial_num_eigs = 40   # wall eigenmodes computed before choosing the num_modes to keep
noise_level = 0.01      # Gaussian noise added to the synthetic signals, as a fraction of each sensor's peak

In [ ]:
# Create OFT environment and JAMfit object
myOFT = OFT_env(nthreads=4)
jam_obj = JAMfit(xml_file, thincurr_meshfile, tokamaker_meshfile, B0, R0, oft_env=myOFT)

## Setting up synthetic sensors
Here we define synthetic magnetic probes using the \ref OpenFUSIONToolkit.ThinCurr.sensor.Mirnov "Mirnov" sensor class in ThinCurr. We then load them into JAMfit with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.set_sensors "set_sensors()". Afterwards \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.setup_JAMfit "setup_JAMfit()" computes the mutual inductance and resistance matrices. Like ThinCurr, JAMfit uses HODLR compression (see \ref doc_tCurr_hodlr for more information).

For this simple case we use 16 Mirnov probes on a circle of minor radius 0.4 m around the torus centre, all at toroidal angle 0, which sits halfway between two ports. Each probe measures the poloidal field, so its normal points along the circle (anticlockwise in the R-Z plane). The angles are offset by half a step so no probe sits exactly on the midplane.

In [ ]:
# --- Poloidal array of Mirnov probes at toroidal angle 0 ---
num_sensors = 16
r_minor = 0.4                                                      # minor radius of the probe circle [m]
theta = (np.arange(num_sensors) + 0.5) * 2.0 * np.pi / num_sensors   # poloidal angle of each probe [rad]
sensors = []
for i, th in enumerate(theta):
    R = R0 + r_minor * np.cos(th)
    Z = r_minor * np.sin(th)
    # point is (x, y, z) with y = 0 (toroidal angle 0); normal is the poloidal direction (-sin, 0, cos)
    sensors.append(Mirnov(np.r_[R, 0.0, Z], np.r_[-np.sin(th), 0.0, np.cos(th)], 'Bp_{0}'.format(i + 1)))

jam_obj.set_sensors(sensors, floops_path=floops_file)
jam_obj.setup_JAMfit(floops_file, mcoil_cache='one_filament_Mcoil.save', lmat_cache='one_filament_L.save')

Plotting the geometry from the TokaMaker mesh: the wall/limiter, the two coils, the filament and the probes.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
jam_obj.mygs.plot_machine(fig, ax)
lim = jam_obj.mygs.lim_contour
ax.plot(lim[:, 0], lim[:, 1], 'k-', label='wall / limiter')
ax.plot(R0 + r_minor * np.cos(theta), r_minor * np.sin(theta), 'gs', ms=5, label='Mirnov probes')
ax.plot(1.0, 0.0, 'r*', ms=12, label='plasma filament')
ax.plot([1.0, 1.0], [0.7, -0.7], 'bo', ms=6, label='upper / lower coils')
ax.set_aspect('equal')
ax.set_xlabel('R [m]')
ax.set_ylabel('Z [m]')
ax.legend(loc='upper left', fontsize=8)
plt.show()

## Making the synthetic data

Here we make the synthetic data with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.gen_synthetic_data "gen_synthetic_data()". It takes the coil currents `[A]` as an array with a leading time column `[s]`, followed by one column per coil in the same order as `oft_in.xml`:

| Column | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| Contents | time | upper coil | lower coil | plasma filament |

In this example the two coils hold a constant 1 kA, and the plasma filament follows a simple waveform:

| Phase | Time | Filament current |
|---|---|---|
| Ramp-up | 0 → 5 ms | 0 → 1 MA |
| Flat-top | 5 → 15 ms | 1 MA |
| Quench | 15 → 20 ms | 1 MA → 0 |
| Decay | 20 → 40 ms | 0 (run continues so the wall currents can decay) |

`gen_synthetic_data()` runs the **full** ThinCurr wall model (every mesh element) and returns the sensor history. These signals act as our "measurements". We then add a small amount of noise to make the reconstruction test more realistic.

In [ ]:
dt = 2.E-4       # time step [s]
nsteps = 200     # 200 steps -> 40 ms

time_pts = np.array([0.0, 0.005, 0.015, 0.020, 0.040])     # [s]
upper_wave = np.array([1.E3, 1.E3, 1.E3, 1.E3, 1.E3])      # upper coil [A]
lower_wave = np.array([1.E3, 1.E3, 1.E3, 1.E3, 1.E3])      # lower coil [A]
fil_wave = np.array([0.0, 1.E6, 1.E6, 0.0, 0.0])           # plasma filament [A]
all_currs = np.column_stack((time_pts, upper_wave, lower_wave, fil_wave))

hist = jam_obj.gen_synthetic_data(all_currs, dt, nsteps, s_freq=50, p_freq=50)

# --- Measurements: (ntimes, num_sensors) in sensor order, plus a little Gaussian noise ---
time_meas = np.asarray(hist['time'])
clean_signals = np.column_stack([hist['Bp_{0}'.format(i + 1)] for i in range(num_sensors)])
rng = np.random.default_rng(0)
noise_sigma = noise_level * np.abs(clean_signals).max(axis=0)   # per-sensor noise level [T]
signals = clean_signals + rng.normal(0.0, 1.0, clean_signals.shape) * noise_sigma

# --- True coil current and Ip at the measurement times (ThinCurr interpolates the waveforms linearly) ---
coil_meas = np.column_stack((np.interp(time_meas, time_pts, upper_wave),
                             np.interp(time_meas, time_pts, lower_wave)))   # (ntimes, num_non_fil_coils) [A]
ip_meas = np.interp(time_meas, time_pts, fil_wave)                      # (ntimes,) [A]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].plot(time_meas * 1E3, ip_meas / 1E6, 'k')
ax[0].set_ylabel('filament current [MA]')
ax[0].grid(True)
ax[1].plot(time_meas * 1E3, signals * 1E3, lw=1)
ax[1].set_ylabel('Mirnov signals [mT]')
ax[1].set_xlabel('time [ms]')
ax[1].grid(True)
plt.show()

## Build the reduced wall model

ThinCurr can build a reduced wall model from a subset of the wall eigenmodes (see \ref doc_tCurr_reduction for more information). JAMfit does this with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.create_from_runTD_top_modes "create_from_runTD_top_modes()", which:

1. computes the first `initial_num_eigs` eigenmodes of the full wall model,
2. runs the same coil waveforms through a preliminary reduced model built from all of those modes,
3. ranks each mode $i$ by its total current over the run, $w_i = \sum_t |I_i(t)|$, and
4. keeps the `num_modes` modes with the largest $w_i$ (the modes that carry the most current and/or live the longest).

The reconstruction then solves for only these few wall-mode amplitudes. With `verbose=True` it plots the amplitude of every mode: the selected modes are in colour and the rest are in grey.

**How many modes?** `num_modes` is a tunable parameter. Here we keep `num_modes = 2`. The reconstructed Ip barely depends on this choice (about 0.2% error for anything from 1 to 10 modes), so we choose by how well the total wall current is reproduced: 2 modes give the smallest wall-current error, while with 3 or more modes the error grows quickly, even without noise, because the extra weakly driven modes soak up the contribution of the modes left out. A second mode also lowers the residual compared with 1 mode. Computing enough initial eigenmodes also matters: with only 20, the second most important mode (index 34 here) is never considered, so we use `initial_num_eigs = 40`.

In [ ]:
reduced_model, prelim_signals, prelim_currents, basis, eig_inds, weights = jam_obj.create_from_runTD_top_modes(
    num_modes, reduced_filename, all_currs, dt, nsteps, initial_num_eigs=initial_num_eigs, verbose=True, s_freq=50, p_freq=5)
plt.xlabel('time [s]')
plt.ylabel('|mode amplitude|')
plt.legend(fontsize=8)
plt.show()
print('Kept wall modes:', eig_inds)

## Running a reconstruction

At each time we reconstruct the wall-mode amplitudes $I_w$ and the filament current $I_f$ with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.run_reconstruction_lstsq "run_reconstruction_lstsq()", by solving the linear system $A x = B$:

$\begin{bmatrix} M_s^T & M_{sc,f}^T \\ 0 & 1 \end{bmatrix} \begin{bmatrix} I_w \\ I_f \end{bmatrix} = \begin{bmatrix} s - M_{sc,c}^T I_c \\ I_p \end{bmatrix}$

| Symbol | Meaning |
|---|---|
| $I_w$, $I_f$ | unknowns: wall-mode amplitudes and filament current |
| $s$ | sensor signals |
| $M_s$ | wall-mode → sensor mutual inductances (reduced model) |
| $M_{sc,f}$, $M_{sc,c}$ | filament → sensor and coil → sensor mutual inductances |
| $I_c$ | known coil currents (upper and lower coil) |
| $I_p$ | measured total plasma current |

The first block row says the wall modes and the filament must reproduce the sensor signals, after subtracting the field from the known coils. The second row says the filament current must add up to the measured Ip. Each sensor row is divided by that sensor's noise `sigma`, and the Ip row is multiplied by `ip_weight` $/\,\hat{I}_p$, where $\hat{I}_p = \max(\lvert I_p \rvert, 1\,\mathrm{kA})$.

With 16 sensors plus the Ip row there are more equations than unknowns (3), so $A x = B$ is solved in the least-squares sense, minimising $\lVert A x - B \rVert^2$.

**Regularization**

In general, with many filaments and wall modes, this problem is ill-posed: many current distributions fit the sensors equally well. `run_reconstruction_lstsq` therefore adds Tikhonov regularization, which penalises large wall amplitudes and filament currents:

$\min_x \; \lVert A x - B \rVert^2 + \lambda_w^2 \lVert I_w \rVert^2 + \lambda_f^2 \lVert I_f \rVert^2$

where $\lambda_w$ = `reg_factor_wall` and $\lambda_f$ = `reg_factor_fil`. Larger values give smaller currents at the cost of a worse fit to the sensors. Our problem is overdetermined, so we set both factors very small.

**Choosing `ip_weight`**

Because the sensor rows are in units of noise and the Ip row is a relative error, `ip_weight = 100` makes a 1% error in Ip cost the same as a 1-σ miss on one sensor. A very large weight forces the filament current to equal the measured Ip, whatever the sensors say. With a good sensor set and wall model the sensors alone already give the right current, so the Ip row is not needed. Below we scan `ip_weight` to see how much the Ip row matters: in this example the flat-top Ip error stays at about 0.2% for any `ip_weight` up to about 100, and only larger weights pull the filament current onto the measured Ip.

**Inputs at each time**

| Input | Description |
|---|---|
| `psi_at_time` | sensor signals at this time, shape `(num_sensors,)` |
| `ip_at_time` | measured total plasma current `[A]` |
| `num_non_fil_coils` | number of non-filament coils listed before the filaments in `oft_in.xml` (here 2) |
| `coil_curr_at_time` | currents in those coils `[A]` |
| `sigma` | noise on each sensor, same units as the signals |
| `ip_weight` | weight of the Ip row |
| `reg_factor_wall`, `reg_factor_fil` | Tikhonov factors $\lambda_w$, $\lambda_f$ |

In [ ]:
sigma = noise_sigma          # noise on each sensor [T], shape (num_sensors,)
reg_fil = 1.E-8              # Tikhonov factor on the filament current
reg_wall = 1.E-8             # Tikhonov factor on the wall mode amplitudes

def reconstruct_all_times(ip_weight):
    '''! Run run_reconstruction_lstsq at every measurement time.

    @param ip_weight float, weight of the Ip constraint row
    @result numpy.ndarray, filament current `[A]`, shape (ntimes,)
    @result numpy.ndarray, wall mode amplitudes, shape (ntimes, num_modes)
    @result numpy.ndarray, residual norm at each time, shape (ntimes,)
    '''
    fil, wall, res = [], [], []
    for t_idx in range(len(time_meas)):
        solution_fil, solution_wall, residual, Ax, B = jam_obj.run_reconstruction_lstsq(
            signals[t_idx, :], ip_meas[t_idx], num_non_fil_coils, coil_meas[t_idx, :],
            ip_weight, sigma, reg_fil, reg_wall)
        fil.append(solution_fil[0])
        wall.append(solution_wall)
        res.append(residual)
    return np.array(fil), np.array(wall), np.array(res)

ip_weight_list = np.logspace(-3, 4, 15)
ip_error_list = []
residual_list = []
for ip_weight in ip_weight_list:
    fil, wall, res = reconstruct_all_times(ip_weight)
    ip_error_list.append(np.mean(np.abs(fil - ip_meas)))
    residual_list.append(np.mean(res))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].loglog(ip_weight_list, residual_list, 'o-')
ax[0].set_xlabel('ip_weight')
ax[0].set_ylabel('mean residual')
ax[0].grid(True)
ax[1].loglog(ip_weight_list, ip_error_list, 'o-')
ax[1].set_xlabel('ip_weight')
ax[1].set_ylabel('mean |I_fil - Ip| [A]')
ax[1].grid(True)
plt.tight_layout()
plt.show()

Now we run the reconstruction with `ip_weight = 1`. At this weight the Ip row barely matters, so this tests how well the sensors alone recover the currents. We compare with the truth:
- the filament current against the true Ip,
- the fitted sensor signals against the measured ones,
- the wall mode amplitudes against those from the preliminary run in `create_from_runTD_top_modes` (that run used `initial_num_eigs` modes, so it is a close stand-in for the true wall currents).

The mean residual in the scan above is about 6, a little above the $\sqrt{16} = 4$ expected if the fit were at the noise level. The extra comes from the wall modes left out of the 2-mode reduced model.

In [ ]:
ip_weight = 1.0
fil, wall, res = reconstruct_all_times(ip_weight)

# --- Fitted sensor signals: wall part + coil part, in [T] (Ms and Msc map currents to sensor signals) ---
Ms = jam_obj.torus_reduced.Ms[:, :num_sensors]            # (num_modes, num_sensors)
Msc = jam_obj.torus_reduced.Msc[:, :num_sensors]          # (num_coils, num_sensors), coils then filament
all_coil_currs = np.column_stack((coil_meas, fil))        # (ntimes, num_coils)
fitted_signals = wall @ Ms + all_coil_currs @ Msc

fig, ax = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
ax[0].plot(time_meas * 1E3, ip_meas / 1E6, 'k--', label='true Ip')
ax[0].plot(time_meas * 1E3, fil / 1E6, 'r', label='reconstructed filament')
ax[0].set_ylabel('current [MA]')
ax[0].legend()
ax[0].grid(True)

for i in [0, num_sensors // 4, num_sensors // 2]:
    line, = ax[1].plot(time_meas * 1E3, signals[:, i] * 1E3, '.', ms=3, label='measured Bp_{0}'.format(i + 1))
    ax[1].plot(time_meas * 1E3, fitted_signals[:, i] * 1E3, '-', color=line.get_color())
ax[1].set_ylabel('signal [mT]')
ax[1].legend(fontsize=8)
ax[1].grid(True)

prelim_time = np.asarray(prelim_currents['time'])
for j, mode in enumerate(eig_inds):
    line, = ax[2].plot(prelim_time * 1E3, prelim_currents['curr'][:, mode], '--', label='mode {0} (preliminary run)'.format(mode))
    ax[2].plot(time_meas * 1E3, wall[:, j], '-', color=line.get_color())
ax[2].set_ylabel('wall mode amplitude')
ax[2].set_xlabel('time [ms]')
ax[2].legend(fontsize=8)
ax[2].grid(True)
plt.tight_layout()
plt.show()

flat_top = (time_meas > 0.006) & (time_meas < 0.015)
print('Mean relative Ip error on the flat top: {0:.2e}'.format(np.mean(np.abs(fil[flat_top] - ip_meas[flat_top]) / ip_meas[flat_top])))

## Next steps
The DIII-D VDE example (\ref doc_tCurr_jamfit_vde_create, then \ref doc_tCurr_jamfit_vde_reconstruct) uses the same steps on a real geometry with many filaments, and demonstrates the other solvers (TSVD/Laplacian), flux probes for the wall psi, and the TokaMaker post-processing (LCFS, q95, li).